# SourceBindings API Query
This notebook connects to the MetadataService API to retrieve and display SourceBinding information by ID.

In [1]:
import requests
import json
import pandas as pd
from IPython.display import display, JSON

In [2]:
# Configuration
BASE_URL = "https://rhnv-webprod.rhnv.hosted/MetadataService/v1/SourceBindings"

# Enter the ID you want to query
binding_id = "12926"  # Replace with your desired ID

# SSL verification (set to False for internal servers with self-signed certificates)
VERIFY_SSL = False

# Suppress SSL warnings when verification is disabled
if not VERIFY_SSL:
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print("Using Windows SSO authentication (current user credentials)")

Using Windows SSO authentication (current user credentials)


In [3]:
def get_source_binding(binding_id):
    """
    Retrieve SourceBinding data from the API using Windows SSO authentication
    
    Args:
        binding_id: The ID of the SourceBinding to retrieve
    
    Returns:
        dict: The JSON response from the API
    """
    url = f"{BASE_URL}({binding_id})"
    
    try:
        # Try using requests_negotiate_sspi for Windows SSO
        from requests_negotiate_sspi import HttpNegotiateAuth
        
        response = requests.get(url, auth=HttpNegotiateAuth(), verify=VERIFY_SSL)
        response.raise_for_status()
        return response.json()
        
    except ImportError:
        print("Installing required package for Windows SSO authentication...")
        print("Please run: pip install requests-negotiate-sspi")
        print("\nAlternatively, trying with requests_ntlm...")
        
        try:
            # Fallback to NTLM with empty credentials (uses current Windows user)
            from requests_ntlm import HttpNtlmAuth
            import getpass
            import os
            
            # Get current Windows username
            current_user = os.environ.get('USERNAME')
            domain = os.environ.get('USERDOMAIN', '')
            
            if domain:
                username = f"{domain}\\{current_user}"
            else:
                username = current_user
            
            print(f"Attempting authentication as: {username}")
            print("Note: You may need to enter your password for NTLM authentication")
            password = getpass.getpass("Enter your password: ")
            
            response = requests.get(url, auth=HttpNtlmAuth(username, password), verify=VERIFY_SSL)
            response.raise_for_status()
            return response.json()
            
        except ImportError:
            print("\nNeither requests-negotiate-sspi nor requests_ntlm is installed.")
            print("Please install one of the following:")
            print("  - For SSO: pip install requests-negotiate-sspi")
            print("  - For NTLM: pip install requests_ntlm")
            return None
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            if hasattr(e, 'response') and e.response is not None:
                print(f"Status Code: {e.response.status_code}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Status Code: {e.response.status_code}")
        return None

In [4]:
# Fetch the data
data = get_source_binding(binding_id)

if data:
    print("✓ Successfully retrieved SourceBinding data")
else:
    print("✗ Failed to retrieve data")

✓ Successfully retrieved SourceBinding data


In [5]:
# Display the complete JSON response
if data:
    print("=" * 80)
    print("COMPLETE JSON RESPONSE")
    print("=" * 80)
    display(JSON(data, expanded=True))

COMPLETE JSON RESPONSE


<IPython.core.display.JSON object>

In [6]:
# Display core fields in a structured format
if data:
    print("\n" + "=" * 80)
    print("CORE FIELDS")
    print("=" * 80)
    
    core_fields = [
        'Id', 'Name', 'Description', 'Status', 'Classification',
        'BindingType', 'GrainName', 'LoadTypeCode',
        'DestinationEntityId', 'DestinationEntity',
        'DataMartId', 'DataMart',
        'SourceConnectionId', 'SourceConnection',
        'GroupingColumn', 'GroupingFormat', 'ContentId'
    ]
    
    for field in core_fields:
        if field in data:
            print(f"{field:25s}: {data[field]}")


CORE FIELDS
Id                       : 12926
Name                     : SummaryMetricRemoteWorkHistory
Description              : None
Status                   : Active
Classification           : Summary
BindingType              : SQL
GrainName                : None
LoadTypeCode             : Full
DestinationEntityId      : 14144
DataMartId               : 558
SourceConnectionId       : 661
GroupingColumn           : None
GroupingFormat           : None
ContentId                : e4e8ed09-3d58-4c6f-bf04-cd3fad9dff7f


In [7]:
# Display AttributeValues if present
if data and 'AttributeValues' in data and data['AttributeValues']:
    print("\n" + "=" * 80)
    print("ATTRIBUTE VALUES")
    print("=" * 80)
    
    if isinstance(data['AttributeValues'], list):
        df_attrs = pd.DataFrame(data['AttributeValues'])
        display(df_attrs)
    else:
        print(data['AttributeValues'])


ATTRIBUTE VALUES


,ObjectId,Id,AttributeName,AttributeValue
0,12926,215322,DisableIncrementalQueryTokenValidation,False
1,12926,215321,IsProtected,False
2,12926,215320,Tags,NaN
3,12926,215319,UserDefinedSQL,/************************* SAM Documentation *...


In [8]:
# Display SourcedByEntities if present
if data and 'SourcedByEntities' in data and data['SourcedByEntities']:
    print("\n" + "=" * 80)
    print("SOURCED BY ENTITIES")
    print("=" * 80)
    
    if isinstance(data['SourcedByEntities'], list):
        df_sourced = pd.DataFrame(data['SourcedByEntities'])
        display(df_sourced)
    else:
        print(data['SourcedByEntities'])

In [9]:
# Display IncrementalConfigurations if present
if data and 'IncrementalConfigurations' in data and data['IncrementalConfigurations']:
    print("\n" + "=" * 80)
    print("INCREMENTAL CONFIGURATIONS")
    print("=" * 80)
    
    if isinstance(data['IncrementalConfigurations'], list):
        df_incremental = pd.DataFrame(data['IncrementalConfigurations'])
        display(df_incremental)
    else:
        print(data['IncrementalConfigurations'])

In [10]:
# Display ObjectRelationships if present
if data and 'ObjectRelationships' in data and data['ObjectRelationships']:
    print("\n" + "=" * 80)
    print("OBJECT RELATIONSHIPS")
    print("=" * 80)
    
    if isinstance(data['ObjectRelationships'], list):
        df_relationships = pd.DataFrame(data['ObjectRelationships'])
        display(df_relationships)
    else:
        print(data['ObjectRelationships'])

In [11]:
# Export to JSON file (optional)
if data:
    output_filename = f"SourceBinding_{binding_id}.json"
    with open(output_filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"\n✓ Data exported to {output_filename}")


✓ Data exported to SourceBinding_12926.json
